In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, TFBertForSequenceClassification, AutoConfig, BertConfig
from io import StringIO
import os

In [4]:
# --- 0. CONFIGURATION ---
MODEL_NAME = 'bert-base-uncased'
HUMAN_MAPPING_FILE = 'CO_POMapping_ELECTIVE4.csv'
PO_FILE_PATH = 'Book1_ELECTIVE4.csv'
CO_FILE_PATH = 'Book2_ELECTIVE4.csv'
MODEL_DIR = './fine_tuned_bert_co_po' # Directory to save the trained model

In [5]:
# Define Labels and their corresponding integer IDs (0, 1, 2)
LABELS = ['I', 'D', 'E']
NUM_LABELS = len(LABELS)
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
ID_TO_LABEL = {i: label for i, label in enumerate(LABELS)}

MAX_SEQ_LENGTH = 128
BATCH_SIZE = 16
NUM_EPOCHS = 10
LEARNING_RATE = 2e-5

In [6]:
# Check for GPU (Recommended for training)
if tf.config.experimental.list_physical_devices('GPU'):
    print("GPU found and being used for training.")
else:
    print("No GPU found. Training will use CPU and may take longer.")

No GPU found. Training will use CPU and may take longer.


In [7]:
# --- 1. DATA PREPARATION FOR FINE-TUNING ---

try:
    # 1.1 Load Mapping (Your Training Data)
    human_mapping_df = pd.read_csv(HUMAN_MAPPING_FILE)
    human_mapping_df.columns = human_mapping_df.columns.str.strip()

    # 1.2 Load COs and POs (Descriptions)
    po_df = pd.read_csv(PO_FILE_PATH)
    po_df.columns = ['PO', 'PO_Description']
    co_df = pd.read_csv(CO_FILE_PATH)
    co_df.columns = ['CO', 'CO_Description']

except FileNotFoundError as e:
    print(f"FATAL ERROR: Required file not found for training or setup: {e}")
    exit()

In [8]:
# Create lookup dictionaries
program_outcomes = dict(zip(po_df['PO'], po_df['PO_Description']))
course_outcomes = dict(zip(co_df['CO'], co_df['CO_Description']))

print(f"Successfully loaded {len(course_outcomes)} COs and {len(program_outcomes)} POs.")

Successfully loaded 4 COs and 15 POs.


In [9]:
# 1.3 Create the BERT Training DataFrame (Sentence Pair Format)
train_data_list = []
co_columns = human_mapping_df['CO'].tolist()

In [10]:
# Get all PO columns (excluding 'CO' and 'CO Description')
po_columns = [col for col in human_mapping_df.columns if col.startswith('PO')]

In [11]:
for index, row in human_mapping_df.iterrows():
    co_id = row['CO']
    # Use the description from the CO file
    co_desc = course_outcomes.get(co_id, None)

    if co_desc is None:
        print(f"Warning: CO ID {co_id} not found in {CO_FILE_PATH}. Skipping.")
        continue

    for po_id in po_columns:
        po_desc = program_outcomes.get(po_id, None)

        # Get the correlation level (I, D, E, or -)
        correlation_str = str(row.get(po_id, '-')).strip().upper()

        # Only train on valid labels (I, D, E). Ignore '-' (No Correlation)
        if correlation_str in LABELS and po_desc is not None:
            train_data_list.append({
                'sentence1': co_desc,
                'sentence2': po_desc,
                'label': LABEL_TO_ID[correlation_str] # Convert 'I'/'D'/'E' to 0/1/2
            })

df_train = pd.DataFrame(train_data_list)
print(f"Created {len(df_train)} training examples.")

Created 60 training examples.


In [12]:
df_train

,sentence1,sentence2,label
0,Understand the concepts of information systems...,"Apply knowledge of computing, science, and mat...",0
1,Understand the concepts of information systems...,Use current best practices and standards in so...,0
2,Understand the concepts of information systems...,Analyze complex computing/IT - related problem...,2
3,Understand the concepts of information systems...,Identify and analyze user needs and take them ...,2
4,Understand the concepts of information systems...,"Design creatively, implement and evaluate diff...",1
5,Understand the concepts of information systems...,Integrate effectively the IT-based solutions i...,1
6,Understand the concepts of information systems...,"Select, adapt and apply appropriate techniques...",1
7,Understand the concepts of information systems...,"Function effectively as individual, or work co...",1
8,Understand the concepts of information systems...,Assist in the creation of an effective IT proj...,1
9,Understand the concepts of information systems...,Communicate effectively in both oral and in wr...,1


In [13]:
# Split data into training and validation sets
train_texts_1, val_texts_1, train_texts_2, val_texts_2, train_labels, val_labels = \
    train_test_split(df_train['sentence1'].tolist(),
                     df_train['sentence2'].tolist(),
                     df_train['label'].tolist(),
                     test_size=0.1,
                     random_state=42,
                     stratify=df_train['label'].tolist())

In [14]:
# --- 2. TOKENIZATION AND DATASET CREATION ---

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

In [15]:
def encode_data(text_list_1, text_list_2, labels):
    encodings = tokenizer(text_list_1, text_list_2,
                          truncation=True,
                          padding='max_length',
                          max_length=MAX_SEQ_LENGTH,
                          return_tensors='tf')

    dataset = tf.data.Dataset.from_tensor_slices((
        {'input_ids': encodings['input_ids'],
         'attention_mask': encodings['attention_mask'],
         'token_type_ids': encodings['token_type_ids']},
        tf.constant(labels)
    )).shuffle(len(labels)).batch(BATCH_SIZE)

    return dataset

In [16]:
train_dataset = encode_data(train_texts_1, train_texts_2, train_labels)
val_dataset = encode_data(val_texts_1, val_texts_2, val_labels)

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


In [17]:
config = BertConfig.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID
)

In [18]:
model = TFBertForSequenceClassification.from_pretrained(MODEL_NAME, config=config, from_pt=True)

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [19]:
# Setup the AdamW optimizer (critical for BERT fine-tuning)
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, epsilon=1e-08)

loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(optimizer=optimizer,
              loss=loss,
              metrics=[tf.keras.metrics.SparseCategoricalAccuracy('accuracy')])

print("\nModel Compiled. Starting Fine-Tuning...")


Model Compiled. Starting Fine-Tuning...


In [20]:
# --- 4. FINE-TUNING THE MODEL ---

# Train the model
history = model.fit(
    train_dataset,
    epochs=NUM_EPOCHS,
    validation_data=val_dataset
)

Epoch 1/10

4/4 [==============================] - 99s 14s/step - loss: 1.1380 - accuracy: 0.3148 - val_loss: 0.9985 - val_accuracy: 0.6667
Epoch 2/10
4/4 [==============================] - 34s 8s/step - loss: 0.9328 - accuracy: 0.6667 - val_loss: 0.8168 - val_accuracy: 0.6667
Epoch 3/10
4/4 [==============================] - 33s 8s/step - loss: 0.7929 - accuracy: 0.6667 - val_loss: 0.6402 - val_accuracy: 0.6667
Epoch 4/10
4/4 [==============================] - 35s 8s/step - loss: 0.7044 - accuracy: 0.6667 - val_loss: 0.5983 - val_accuracy: 0.6667
Epoch 5/10
4/4 [==============================] - 30s 7s/step - loss: 0.6406 - accuracy: 0.7593 - val_loss: 0.5329 - val_accuracy: 0.8333
Epoch 6/10
4/4 [==============================] - 31s 7s/step - loss: 0.5688 - accuracy: 0.7963 - val_loss: 0.5295 - val_accuracy: 0.8333
Epoch 7/10
4/4 [==============================] - 30s 7s/step - loss: 0.5032 - accuracy: 0.8519 - val_loss: 0.5768 - val_accuracy: 0.6667
Epoch 8/10
4/4 [================

In [21]:
# --- 5. SAVING THE FINE-TUNED MODEL ---

os.makedirs(MODEL_DIR, exist_ok=True)
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print(f"\nFine-tuned model saved to: {MODEL_DIR}")
print("="*80)


Fine-tuned model saved to: ./fine_tuned_bert_co_po


In [22]:
# --- 6. INFERENCE (Prediction) USING THE FINE-TUNED MODEL ---

# Update the MODEL_NAME to the directory where the fine-tuned model was saved
PREDICTION_MODEL_PATH = MODEL_DIR # './fine_tuned_bert_co_po'

In [23]:
# 6.1 Reload the Fine-Tuned Model and Tokenizer
print(f"\nLoading fine-tuned model from: {PREDICTION_MODEL_PATH}")
tokenizer_tuned = BertTokenizer.from_pretrained(PREDICTION_MODEL_PATH)
config_tuned = AutoConfig.from_pretrained(PREDICTION_MODEL_PATH)


Loading fine-tuned model from: ./fine_tuned_bert_co_po


In [32]:
# Load the model weights and configuration
model_tuned = TFBertForSequenceClassification.from_pretrained(PREDICTION_MODEL_PATH, config=config_tuned)

Some layers from the model checkpoint at ./fine_tuned_bert_co_po were not used when initializing TFBertForSequenceClassification: ['dropout_37']
- This IS expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertForSequenceClassification were initialized from the model checkpoint at ./fine_tuned_bert_co_po.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForSequenceClassification for predictions without further training.


In [25]:
def get_co_po_mapping_tuned(co_text, po_text, model, tokenizer, config):
    """
    Feeds a CO-PO pair to the FINE-TUNED BERT model and returns the predicted
    correlation string and the prediction probabilities for all labels.
    """
    # 1. Preprocess input
    inputs = tokenizer(
        co_text,
        po_text,
        truncation=True,
        padding='max_length',
        max_length=MAX_SEQ_LENGTH,
        return_tensors='tf'
    )

    # 2. Get prediction (using training=False for inference)
    # The output is a TFSequenceClassifierOutput object; access logits directly
    outputs = model(inputs, training=False)
    logits = outputs.logits

    # Apply softmax to convert logits to probabilities
    probabilities = tf.nn.softmax(logits, axis=1).numpy()[0]

    # 3. Find the predicted class ID (0, 1, or 2)
    predicted_class_id = tf.argmax(logits, axis=1).numpy()[0]

    # 4. Map ID to I, D, or E using the config dictionary
    predicted_label = config.id2label.get(predicted_class_id, '-')

    # Return both the predicted label and the probabilities
    return predicted_label, probabilities

In [26]:
# 6.2 Generate the new Matrix using the Fine-Tuned Model
df_matrix_tuned = co_df[['CO', 'CO_Description']].copy() # Use the initial COs

In [27]:
# Re-list PO columns from the initial PO file for matrix creation
po_columns = list(program_outcomes.keys())

# Lists to store probabilities for each PO column
prob_cols_I = {}
prob_cols_D = {}
prob_cols_E = {}

In [28]:
for po_id in po_columns:
    predicted_labels = []
    probs_I = []
    probs_D = []
    probs_E = []

    for co_description in df_matrix_tuned['CO_Description']:
        predicted_label, probabilities = get_co_po_mapping_tuned(
            co_text=co_description,
            po_text=program_outcomes[po_id],
            model=model_tuned,
            tokenizer=tokenizer_tuned,
            config=config_tuned
        )
        predicted_labels.append(predicted_label)
        probs_I.append(probabilities[LABEL_TO_ID['I']])
        probs_D.append(probabilities[LABEL_TO_ID['D']])
        probs_E.append(probabilities[LABEL_TO_ID['E']])

    df_matrix_tuned[po_id] = predicted_labels
    prob_cols_I[f'{po_id}_Prob_I'] = probs_I
    prob_cols_D[f'{po_id}_Prob_D'] = probs_D
    prob_cols_E[f'{po_id}_Prob_E'] = probs_E


In [29]:
# Add probability columns to the matrix DataFrame
for col_name, values in prob_cols_I.items():
    df_matrix_tuned[col_name] = values
for col_name, values in prob_cols_D.items():
    df_matrix_tuned[col_name] = values
for col_name, values in prob_cols_E.items():
    df_matrix_tuned[col_name] = values

In [30]:
print("\n--- Generated CO-PO Articulation Matrix (Using FINE-TUNED BERT) ---")
print("Includes predicted labels and probabilities.")
# Displaying only the predicted labels for clarity in the main matrix view
display(df_matrix_tuned.drop(columns=[col for col in df_matrix_tuned.columns if '_Prob_' in col]).set_index('CO'))


--- Generated CO-PO Articulation Matrix (Using FINE-TUNED BERT) ---
Includes predicted labels and probabilities.


,CO_Description,PO1,PO2,PO3,PO4,PO5,PO6,PO7,PO8,PO9,PO10,PO11,PO12,PO13,PO14,PO15
CO,,,,,,,,,,,,,,,,
CO1,Understand the concepts of information systems...,D,D,E,E,D,D,D,D,D,D,D,D,D,D,D
CO2,Apply software methodologies and techniques pr...,D,D,E,E,E,E,D,D,D,E,D,E,D,D,D
CO3,Determine and mode information system requirem...,D,D,E,E,D,E,D,D,D,E,D,E,D,D,D
CO4,Develop and manage an information system project.,D,D,E,E,D,D,D,D,D,E,D,D,D,D,D


In [31]:
print("\n--- HUMAN CO-PO MAPPING ---")
display(human_mapping_df.drop(columns=['CO_Description']).set_index('CO')) # Display human mapping as a table


--- HUMAN CO-PO MAPPING ---


,PO1,PO2,PO3,PO4,PO5,PO6,PO7,PO8,PO9,PO10,PO11,PO12,PO13,PO14,PO15
CO,,,,,,,,,,,,,,,
CO1,I,I,E,E,D,D,D,D,D,D,D,D,D,D,D
CO2,D,D,E,E,E,E,D,D,D,E,D,E,D,E,D
CO3,D,D,E,E,D,E,D,D,D,E,D,E,D,E,D
CO4,D,D,E,E,D,D,D,D,D,E,D,D,D,D,D


In [33]:
# --- Generate NLP Data Mapping using BERT Predictions with Probabilities ---

# Use the initial COs and POs to create pairs
co_list = df_matrix_tuned['CO'].tolist() # Use COs from the matrix (includes descriptions)
po_list = list(program_outcomes.keys()) # Use PO IDs from the initial PO file


In [34]:
nlp_data_mapping_list = []

for co_id in co_list:
    co_desc = course_outcomes.get(co_id, None)
    if co_desc is None:
        continue # Skip if CO description not found

    for po_id in po_list:
        po_desc = program_outcomes.get(po_id, None)
        if po_desc is None:
            continue # Skip if PO description not found

        # Get the predicted classification and probabilities from the fine-tuned model
        predicted_label, probabilities = get_co_po_mapping_tuned(
            co_text=co_desc,
            po_text=po_desc,
            model=model_tuned,
            tokenizer=tokenizer_tuned,
            config=config_tuned
        )

        # Find the human-assigned label for this CO-PO pair
        # Need to handle the case where a PO column might not exist in human_mapping_df
        human_label = human_mapping_df[human_mapping_df['CO'] == co_id].get(po_id, pd.NA).iloc[0]


        # Get the probability the model assigned to the human label
        human_label_prob = None
        if pd.notna(human_label) and human_label in LABEL_TO_ID:
             human_label_prob = probabilities[LABEL_TO_ID[human_label]]


        nlp_data_mapping_list.append({
            'CO': co_id,
            'PO': po_id,
            'CO_Description': co_desc,
            'PO_Description': po_desc,
            'Human_Label': human_label, # Add human label for comparison
            'Predicted_Label': predicted_label,
            'Prob_I': probabilities[LABEL_TO_ID['I']],
            'Prob_D': probabilities[LABEL_TO_ID['D']],
            'Prob_E': probabilities[LABEL_TO_ID['E']],
            'Human_Label_Prob': human_label_prob # Probability of the human-assigned label
        })

df_nlp_data_mapping = pd.DataFrame(nlp_data_mapping_list)

In [35]:
print("\n--- Generated NLP Data Mapping with BERT Predictions and Probabilities ---")
print("Includes human label, predicted label, and probabilities for each classification.")
display(df_nlp_data_mapping)


--- Generated NLP Data Mapping with BERT Predictions and Probabilities ---
Includes human label, predicted label, and probabilities for each classification.


,CO,PO,CO_Description,PO_Description,Human_Label,Predicted_Label,Prob_I,Prob_D,Prob_E,Human_Label_Prob
0,CO1,PO1,Understand the concepts of information systems...,"Apply knowledge of computing, science, and mat...",I,D,0.145380,0.794030,0.060590,0.145380
1,CO1,PO2,Understand the concepts of information systems...,Use current best practices and standards in so...,I,D,0.100292,0.842679,0.057030,0.100292
2,CO1,PO3,Understand the concepts of information systems...,Analyze complex computing/IT - related problem...,E,E,0.109508,0.141164,0.749327,0.749327
3,CO1,PO4,Understand the concepts of information systems...,Identify and analyze user needs and take them ...,E,E,0.106146,0.150695,0.743160,0.743160
4,CO1,PO5,Understand the concepts of information systems...,"Design creatively, implement and evaluate diff...",D,D,0.053191,0.891989,0.054820,0.891989
5,CO1,PO6,Understand the concepts of information systems...,Integrate effectively the IT-based solutions i...,D,D,0.033575,0.894923,0.071501,0.894923
6,CO1,PO7,Understand the concepts of information systems...,"Select, adapt and apply appropriate techniques...",D,D,0.093262,0.853194,0.053545,0.853194
7,CO1,PO8,Understand the concepts of information systems...,"Function effectively as individual, or work co...",D,D,0.091309,0.852919,0.055773,0.852919
8,CO1,PO9,Understand the concepts of information systems...,Assist in the creation of an effective IT proj...,D,D,0.062674,0.890532,0.046794,0.890532
9,CO1,PO10,Understand the concepts of information systems...,Communicate effectively in both oral and in wr...,D,D,0.052905,0.629206,0.317889,0.629206


In [36]:
# --- Display BERT's Generated Classification Matrix ---

print("\n--- Generated CO-PO Articulation Matrix (Using FINE-TUNED BERT) ---")
print("This matrix shows the classifications predicted by the BERT model.")
display(df_matrix_tuned.set_index('CO'))


--- Generated CO-PO Articulation Matrix (Using FINE-TUNED BERT) ---
This matrix shows the classifications predicted by the BERT model.


,CO_Description,PO1,PO2,PO3,PO4,PO5,PO6,PO7,PO8,PO9,...,PO6_Prob_E,PO7_Prob_E,PO8_Prob_E,PO9_Prob_E,PO10_Prob_E,PO11_Prob_E,PO12_Prob_E,PO13_Prob_E,PO14_Prob_E,PO15_Prob_E
CO,,,,,,,,,,,,,,,,,,,,,
CO1,Understand the concepts of information systems...,D,D,E,E,D,D,D,D,D,...,0.071501,0.053545,0.055773,0.046794,0.317889,0.047539,0.144342,0.050316,0.059950,0.046209
CO2,Apply software methodologies and techniques pr...,D,D,E,E,E,E,D,D,D,...,0.719920,0.055084,0.065328,0.052428,0.737129,0.055927,0.709023,0.051583,0.298613,0.048837
CO3,Determine and mode information system requirem...,D,D,E,E,D,E,D,D,D,...,0.712997,0.054506,0.062994,0.053095,0.729601,0.056800,0.713345,0.050805,0.316664,0.048840
CO4,Develop and manage an information system project.,D,D,E,E,D,D,D,D,D,...,0.173255,0.054338,0.058490,0.050426,0.702524,0.051446,0.091811,0.050997,0.069108,0.048211


In [37]:
# --- 7. COMPARING HUMAN MAPPING AND MODEL PREDICTIONS ---

# Align columns to ensure proper comparison
human_mapping_aligned = human_mapping_df.set_index('CO').drop(columns=['CO_Description'])

In [38]:
# Ensure model_predictions_aligned only contains the predicted labels for the PO columns
# that are present in the human_mapping_aligned DataFrame.
model_predictions_aligned = df_matrix_tuned.set_index('CO')[human_mapping_aligned.columns]


In [39]:
# Find the difference - this will show True where values are different
# We need to handle NaN values gracefully, so fill NaN with a placeholder for comparison
difference = (human_mapping_aligned.fillna('NaN') != model_predictions_aligned.fillna('NaN'))


In [40]:
# Create a dataframe to store the comparison results
comparison_df = pd.DataFrame(index=human_mapping_aligned.index, columns=human_mapping_aligned.columns)


In [41]:

for col in comparison_df.columns:
    for row in comparison_df.index:
        human_val = human_mapping_aligned.loc[row, col]
        model_val = model_predictions_aligned.loc[row, col]

        if pd.isna(human_val) and pd.isna(model_val):
            comparison_df.loc[row, col] = '-' # Or some other indicator for both NaN
        elif pd.isna(human_val) or pd.isna(model_val) or human_val != model_val:
            # Highlight differences: show human (model)
            comparison_df.loc[row, col] = f"{human_val} ({model_val})"
        else:
            # No difference, just show the value
            comparison_df.loc[row, col] = human_val

In [42]:

print("\n--- Comparison: Human Mapping vs. Model Predictions (Human (Model)) ---")
print("Differences are highlighted as 'Human Value (Model Value)'")
print("'-' indicates both human and model are NaN (no correlation)")
display(comparison_df)


--- Comparison: Human Mapping vs. Model Predictions (Human (Model)) ---
Differences are highlighted as 'Human Value (Model Value)'
'-' indicates both human and model are NaN (no correlation)


,PO1,PO2,PO3,PO4,PO5,PO6,PO7,PO8,PO9,PO10,PO11,PO12,PO13,PO14,PO15
CO,,,,,,,,,,,,,,,
CO1,I (D),I (D),E,E,D,D,D,D,D,D,D,D,D,D,D
CO2,D,D,E,E,E,E,D,D,D,E,D,E,D,E (D),D
CO3,D,D,E,E,D,E,D,D,D,E,D,E,D,E (D),D
CO4,D,D,E,E,D,D,D,D,D,E,D,D,D,D,D


In [43]:
# --- Evaluate Model Performance on the Validation Set ---

# The `model.fit` output already contains the validation accuracy,
# but we can evaluate explicitly to see the final performance on the validation set.
print("\nEvaluating model on the validation dataset...")
loss, accuracy = model.evaluate(val_dataset)

print(f"Validation Loss: {loss:.4f}")
print(f"Validation Accuracy: {accuracy:.4f}")


Evaluating model on the validation dataset...
1/1 [==============================] - 5s 5s/step - loss: 0.3624 - accuracy: 0.8333
Validation Loss: 0.3624
Validation Accuracy: 0.8333
